## Configuration

Please specify the path to your configuration file below.

In [ ]:
from pathlib import Path

from swissclim_evaluations.cli import _load_yaml, prepare_datasets, run_selected
from swissclim_evaluations.helpers import display_outputs


In [ ]:
if "config_path_str" not in globals():
    config_path_str = "config/example_config.yaml"

In [ ]:
# Locate configuration relative to the notebook location
cfg_path = None

if config_path_str:
    candidate = Path(config_path_str)
    if candidate.is_absolute():
        if candidate.is_file():
            cfg_path = candidate
    else:
        # Try relative to cwd
        if (Path.cwd() / candidate).is_file():
            cfg_path = (Path.cwd() / candidate).resolve()
        # Try relative to parent (if running from notebooks dir)
        elif (Path.cwd().parent / candidate).is_file():
            cfg_path = (Path.cwd().parent / candidate).resolve()
        # Try relative to grandparent
        elif (Path.cwd().parent.parent / candidate).is_file():
            cfg_path = (Path.cwd().parent.parent / candidate).resolve()

    if cfg_path is None:
        print(f"Warning: Config file not found at {candidate} (checked relative to cwd and parents). Falling back to auto-discovery.")
    else:
        print(f"Using configuration: {cfg_path}")

if cfg_path is None:
    # Try to find full_verification_run_prob.yaml first, then fall back to example_config.yaml
    config_names = ["full_verification_run_prob.yaml", "example_config.yaml"]

    for name in config_names:
        for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
            candidate = base / "config" / name
            if candidate.is_file():
                cfg_path = candidate
                print(f"Using configuration: {cfg_path}")
                break
        if cfg_path:
            break

if cfg_path is None:
    raise FileNotFoundError(
        "Could not find config file (full_verification_run_prob.yaml or example_config.yaml) in cwd, parent, or grandparent directories."
    )

# Load configuration via project helper to keep parity with CLI
cfg = _load_yaml(cfg_path)

# Prepare datasets using the CLI pipeline (handles selection, alignment, ensemble policy)
ds_target, ds_prediction, ds_std, ds_prediction_std = prepare_datasets(cfg)

In [ ]:
# Ensure the config has probabilistic enabled
cfg_modules = cfg.get("modules", {})
if not cfg_modules.get("probabilistic", False):
    print("Enabling modules.probabilistic in-memory for this run…")
    cfg_modules["probabilistic"] = True
    cfg["modules"] = cfg_modules

# Set up output directory from config relative to project root (parent of notebooks)
# Project root is the parent of the config directory
try:
    # Find the index of the 'config' directory in the path
    config_idx = cfg_path.parts.index('config')
    # The project root is the path up to the 'config' directory
    project_root = Path(*cfg_path.parts[:config_idx]).resolve()
except ValueError:
    # Fallback: assume standard structure project/config/file.yaml if 'config' not found in path
    project_root = cfg_path.parent.parent.resolve()

cfg_output = cfg.get("paths", {}).get("output_root", "output/notebook_prob")
out_root = (
    (project_root / cfg_output).resolve()
    if not cfg_output.startswith("/")
    else Path(cfg_output)
)
out_root.mkdir(parents=True, exist_ok=True)
print(f"Project root: {project_root}")
print(f"Output root: {out_root}")

# Optional: Execute the full pipeline (data prep, metrics, plots) using CLI logic.
# This will generate all configured artifacts to out_root respecting cfg['modules'] flags
# and cfg['plotting']['output_mode'] (e.g., 'npz' to export numeric data only).
RUN_ALL = False  # Set to True to run everything end-to-end automatically.

if RUN_ALL:
    run_selected(cfg)
    print(f"All selected modules executed. Outputs under: {out_root}")

## CRPS Maps
Continuous Ranked Probability Score (CRPS) maps.
Shows the spatial distribution of the CRPS metric. Lower is better.

**Relevant Code:**
*   [`run_probabilistic`](../src/swissclim_evaluations/metrics/probabilistic/driver.py#L27): Computes CRPS summary and fields.
*   [`plot_probabilistic`](../src/swissclim_evaluations/metrics/probabilistic/driver.py#L61): Generates the CRPS map plots.

In [ ]:
# Display CRPS Maps (optional — only present when output_mode includes 'plot')
print("Displaying CRPS Maps...")
display_outputs(out_root / "probabilistic", pattern_img="crps_map_*.png", pattern_csv="crps_summary*.csv")

In [ ]:
# Per-lead CRPS (line plots + by-lead CSVs)
display_outputs(out_root / "probabilistic", pattern_img="crps_line_*.png", pattern_csv="crps_line_*_by_lead*.csv")

In [ ]:
# CRPS Spatial Maps — per-variable map PNGs (requires output_mode: plot or both)
display_outputs(out_root / "probabilistic", pattern_img="crps_map_*.png", pattern_csv=None)


## PIT Histograms
Probability Integral Transform (PIT) histograms.
Checks for calibration. A flat histogram indicates a well-calibrated ensemble.
U-shape indicates under-dispersion (overconfident).
Inverted U-shape indicates over-dispersion (underconfident).

**Relevant Code:**
*   [`probability_integral_transform`](../src/swissclim_evaluations/metrics/probabilistic/calc.py#L98): Computes the PIT values.
*   [`plot_probabilistic`](../src/swissclim_evaluations/metrics/probabilistic/driver.py#L61): Generates the PIT histogram plots.

In [ ]:
# Display PIT Histograms
print("Displaying PIT Histograms...")
display_outputs(out_root / "probabilistic", pattern_img="pit_hist_*.png", pattern_csv=None, exclude_pattern="grid")

In [ ]:
# Multi-Lead Histograms (Grids) — one panel per lead time
display_outputs(out_root / "probabilistic", pattern_img="pit_hist_*grid*.png", pattern_csv=None)

## Spread-Skill Ratio (SSR)
Compares the ensemble spread (standard deviation) with the RMSE of the ensemble mean.
For a well-calibrated ensemble, spread should match the error.

**Relevant Code:**
*   [`run_probabilistic`](../src/swissclim_evaluations/metrics/probabilistic/driver.py#L27): Calculates Spread-Skill Ratio.


In [ ]:
# Display SSR summary tables
print("Displaying SSR summary tables...")
display_outputs(out_root / "probabilistic", pattern_img=None, pattern_csv="ssr_summary*.csv")

In [ ]:
# SSR figures + per-lead CSVs
display_outputs(out_root / "probabilistic", pattern_img="ssr_*.png", pattern_csv="ssr_line_*_by_lead*.csv")

In [ ]:
# SSR Spatial Maps — per-variable map PNGs (requires output_mode: plot or both)
display_outputs(out_root / "probabilistic", pattern_img="ssr_map_*.png", pattern_csv=None)
